# CCA 3 Computer Vision Assignment
## Practical Implementation: Restoration, Features, Recognition & Motion Estimation
**Environment:** Google Colab / Local Python 3.10+

---

### Overview of Tasks
1. **Task 1: Image Restoration and Segmentation** - Synthetic noise injection (Gaussian, Salt-and-Pepper), spatial filtering (Gaussian, Median), PSNR/SSIM evaluation, thresholding (Global, Otsu, Adaptive), morphology, Canny edges, and Watershed segmentation.
2. **Task 2: Feature Extraction and Matching** - Histogram of Oriented Gradients (HOG), SIFT keypoints & descriptors, Lowe's ratio test matching, and rotation/scale invariance tests.
3. **Task 3: Object Recognition** - Classical Viola-Jones Haar Cascade vs. Modern Ultralytics YOLOv8 deep learning detector.
4. **Task 4: Video Motion Estimation** - Temporal Frame Differencing, MOG2 Background Subtraction, and Lucas-Kanade Optical Flow.


---
## Section 2: Environment Setup and Directory Initialization
Verifies dependencies, sets random seeds for reproducibility, and constructs output folders.


In [3]:
# Cell 2.1: Optional package installation (Recommended for Google Colab)
# Note: Pinning opencv-python < 5.0.0.0 ensures stable cv2.CascadeClassifier support
# !pip install -q "opencv-python<5.0.0.0" scikit-image ultralytics pandas matplotlib scipy


In [4]:
# Cell 2.2: Standard and Scientific Library Imports
import os
import sys
import glob
import time
import shutil
import zipfile
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display, HTML
except ImportError:
    display = print
# Check whether running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

# Computer Vision & Image Processing
try:
    import cv2
    cv2_version = cv2.__version__
except ImportError as e:
    cv2 = None
    cv2_version = f"Not Installed ({e})"

# Scikit-Image
try:
    import skimage
    from skimage import metrics as ski_metrics
    from skimage.feature import hog
    skimage_version = skimage.__version__
except ImportError as e:
    skimage = None
    ski_metrics = None
    hog = None
    skimage_version = f"Not Installed ({e})"

# SciPy
try:
    import scipy
    from scipy import ndimage
    scipy_version = scipy.__version__
except ImportError as e:
    scipy = None
    ndimage = None
    scipy_version = f"Not Installed ({e})"

# Ultralytics YOLO
try:
    import ultralytics
    from ultralytics import YOLO
    ultralytics_version = ultralytics.__version__
except ImportError as e:
    ultralytics = None
    YOLO = None
    ultralytics_version = f"Not Installed ({e})"

print("=" * 65)
print("ENVIRONMENT & DEPENDENCY STATUS:")
print(f"  Python Runtime      : {sys.version.split()[0]}")
print(f"  Execution Context   : {'Google Colab' if IN_COLAB else 'Local Workstation / Jupyter'}")
print(f"  NumPy Version       : {np.__version__}")
print(f"  Pandas Version      : {pd.__version__}")
print(f"  OpenCV Version      : {cv2_version}")
print(f"  scikit-image Version: {skimage_version}")
print(f"  SciPy Version       : {scipy_version}")
print(f"  Ultralytics Version : {ultralytics_version}")
print("=" * 65)


ModuleNotFoundError: No module named 'numpy'

In [ ]:
# Cell 2.3: Reproducible Configuration & Output Directory Creation
# Set random seeds for deterministic noise injection and experiments
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Define project base directory structure
BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, "data")
IMAGES_DIR = os.path.join(DATA_DIR, "images")
VIDEOS_DIR = os.path.join(DATA_DIR, "videos")

OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")
TASK1_REST_DIR = os.path.join(OUTPUT_DIR, "task1_restoration")
TASK1_SEG_DIR = os.path.join(OUTPUT_DIR, "task1_segmentation")
TASK2_DIR = os.path.join(OUTPUT_DIR, "task2_features")
TASK3_DIR = os.path.join(OUTPUT_DIR, "task3_object_recognition")
TASK4_DIR = os.path.join(OUTPUT_DIR, "task4_motion_estimation")
METRICS_DIR = os.path.join(OUTPUT_DIR, "metrics")

ALL_PROJECT_DIRS = [
    DATA_DIR, IMAGES_DIR, VIDEOS_DIR, OUTPUT_DIR,
    TASK1_REST_DIR, TASK1_SEG_DIR, TASK2_DIR,
    TASK3_DIR, TASK4_DIR, METRICS_DIR
]

for directory in ALL_PROJECT_DIRS:
    os.makedirs(directory, exist_ok=True)

print("[✓] Project directory hierarchy verified:")
for d in ALL_PROJECT_DIRS:
    rel_path = os.path.relpath(d, BASE_DIR)
    print(f"    - {rel_path}/")


In [ ]:
# Cell 2.4: Matplotlib Styling & Reusable Visualization Helpers
# Enable inline interactive rendering
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    pass

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#333333'
plt.rcParams['axes.linewidth'] = 0.8

def display_images(images, titles, rows=1, cols=None, figsize=(14, 5), cmap='gray'):
    """Reusable plotting helper for displaying a row/grid of images."""
    n = len(images)
    if cols is None:
        cols = n
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    if n == 1:
        axes = [axes]
    else:
        axes = axes.flatten() if hasattr(axes, 'flatten') else axes

    for idx in range(len(axes)):
        if idx < n:
            img = images[idx]
            if len(img.shape) == 2 or img.shape[2] == 1:
                axes[idx].imshow(img, cmap=cmap)
            else:
                axes[idx].imshow(img)
            axes[idx].set_title(titles[idx], fontsize=11, fontweight='bold', pad=8)
            axes[idx].axis('off')
        else:
            axes[idx].axis('off')
    plt.tight_layout()
    plt.show()

def save_and_display(fig, filepath, title="Visualization"):
    """Helper to persist figure to disk and render cleanly."""
    fig.savefig(filepath, bbox_inches='tight', dpi=150)
    print(f"[Saved] {os.path.basename(filepath)} -> {filepath}")
    plt.show()


---
## Section 3: Input Upload, Discovery, and File Validation
Upload your images and video directly using the file uploader (Cell 3.1) or place them in `data/images/` and `data/videos/`.


In [ ]:
# Cell 3.1: Method A - Interactive Colab File Uploader
def colab_upload_helper():
    """Uploads images and videos directly in Google Colab and sorts them into data/ folders."""
    if IN_COLAB:
        from google.colab import files
        print("[*] Launching Google Colab file upload widget...")
        print("    Please select your images (JPG, PNG) and/or video (MP4) from your computer:")
        uploaded = files.upload()
        for filename in uploaded.keys():
            ext = os.path.splitext(filename)[1].lower()
            if ext in ['.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp']:
                dest = os.path.join(IMAGES_DIR, filename)
                shutil.move(filename, dest)
                print(f"    [+] Saved Image: {filename} -> {os.path.relpath(dest, BASE_DIR)}")
            elif ext in ['.mp4', '.avi', '.mov', '.mkv']:
                dest = os.path.join(VIDEOS_DIR, filename)
                shutil.move(filename, dest)
                print(f"    [+] Saved Video: {filename} -> {os.path.relpath(dest, BASE_DIR)}")
            else:
                print(f"    [?] Unrecognized format: {filename}")
    else:
        print("[Info] Not in Google Colab environment. Place your files in:")
        print(f"       Images: {IMAGES_DIR}")
        print(f"       Videos: {VIDEOS_DIR}")

# In Google Colab: automatically prompt for upload if no image files exist yet
if IN_COLAB:
    existing_imgs = [f for f in os.listdir(IMAGES_DIR) if not f.startswith('.') and f != '.gitkeep']
    if len(existing_imgs) == 0:
        print("[!] No images found in Google Colab virtual session yet.")
        colab_upload_helper()
    else:
        print(f"[✓] Found {len(existing_imgs)} image(s) already in 'data/images/'.")

# To upload additional files at any time, run:
# colab_upload_helper()


In [ ]:
# Cell 3.3: Automatic File Discovery & Selection Configuration
# Valid file extensions
IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp')
VIDEO_EXTENSIONS = ('.mp4', '.avi', '.mov', '.mkv')

def discover_files(directory, extensions):
    """Scans a directory for non-hidden files matching allowed extensions."""
    found = []
    for f in sorted(os.listdir(directory)):
        if f.startswith('.') or f == '.gitkeep':
            continue
        if os.path.splitext(f)[1].lower() in extensions:
            found.append(os.path.join(directory, f))
    return found

discovered_images = discover_files(IMAGES_DIR, IMAGE_EXTENSIONS)
discovered_videos = discover_files(VIDEOS_DIR, VIDEO_EXTENSIONS)

print("=" * 65)
print(f"DISCOVERED INPUT ASSETS:")
print(f"Images in 'data/images/' ({len(discovered_images)} found):")
if discovered_images:
    for i, p in enumerate(discovered_images):
        print(f"  [{i}] {os.path.basename(p)}")
else:
    print("  [Notice] No images found yet. Please place 1 or 2 images in 'data/images/'.")

print(f"\nVideos in 'data/videos/' ({len(discovered_videos)} found):")
if discovered_videos:
    for i, p in enumerate(discovered_videos):
        print(f"  [{i}] {os.path.basename(p)}")
else:
    print("  [Notice] No videos found yet. Please place 1 video in 'data/videos/'.")
print("=" * 65)

# =====================================================================
# ACTIVE FILE CONFIGURATION (MODIFY AS NEEDED)
# =====================================================================
# Set image_path_1: Used for Task 1 (Restoration & Segmentation), Task 2, Task 3
image_path_1 = discovered_images[0] if len(discovered_images) > 0 else None

# Set image_path_2: Used for Task 2 (Two-image SIFT Feature Matching)
image_path_2 = discovered_images[1] if len(discovered_images) > 1 else (discovered_images[0] if len(discovered_images) > 0 else None)

# Set video_path: Used for Task 4 (Video Motion Estimation)
video_path = discovered_videos[0] if len(discovered_videos) > 0 else None

print("\nACTIVE PIPELINE CONFIGURATION:")
print(f"  image_path_1 (Primary Image)   : {image_path_1}")
print(f"  image_path_2 (Secondary Image) : {image_path_2}")
print(f"  video_path   (Target Video)    : {video_path}")


In [ ]:
# Cell 3.4: File Integrity & Input Metadata Validation
def validate_image_input(filepath, label="Image"):
    """Validates file existence, OpenCV decoding, and prints dimension metadata."""
    if filepath is None or not os.path.exists(filepath):
        print(f"[!] {label} Pending: File not found at '{filepath}'.")
        print(f"    Please provide a valid image path to execute {label} experiments.")
        return None, None
    
    bgr = cv2.imread(filepath)
    if bgr is None:
        print(f"[!] OpenCV Decoding Error: Unable to read '{filepath}'. Verify format/codec.")
        return None, None
    
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    h, w, c = rgb.shape
    print(f"[✓] {label} Validated: '{os.path.basename(filepath)}'")
    print(f"    Resolution : {w} x {h} (Width x Height)")
    print(f"    Channels   : {c} (RGB)")
    print(f"    Data Type  : {rgb.dtype}")
    print(f"    Range      : [{rgb.min()}, {rgb.max()}]")
    return bgr, rgb

def validate_video_input(filepath):
    """Validates video file access and extracts metadata without reading all frames into RAM."""
    if filepath is None or not os.path.exists(filepath):
        print(f"[!] Video Input Pending: File not found at '{filepath}'.")
        print("    Please provide a valid video file in 'data/videos/' for Task 4 experiments.")
        return None
    
    cap = cv2.VideoCapture(filepath)
    if not cap.isOpened():
        print(f"[!] OpenCV Video Capture Error: Unable to open '{filepath}'.")
        return None
    
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    fps = fps if (fps is not None and fps > 0 and not math.isnan(fps)) else 25.0
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = frame_count / fps if frame_count > 0 else 0.0
    cap.release()
    
    meta = {
        'filepath': filepath,
        'filename': os.path.basename(filepath),
        'width': w,
        'height': h,
        'fps': fps,
        'frames': frame_count,
        'duration': duration
    }
    print(f"[✓] Video Validated: '{meta['filename']}'")
    print(f"    Resolution : {w} x {h}")
    print(f"    Frame Rate : {fps:.2f} FPS")
    print(f"    Frame Count: {frame_count} frames")
    print(f"    Duration   : {duration:.2f} seconds")
    return meta

# Validate primary assets
primary_bgr, primary_rgb = validate_image_input(image_path_1, "Primary Image (image_path_1)")
secondary_bgr, secondary_rgb = validate_image_input(image_path_2, "Secondary Image (image_path_2)")
video_meta = validate_video_input(video_path)

# Immediate visual confirmation of loaded assets
preview_imgs, preview_titles = [], []
if primary_rgb is not None:
    preview_imgs.append(primary_rgb)
    preview_titles.append(f"Image 1: {os.path.basename(image_path_1)}")
if secondary_rgb is not None:
    preview_imgs.append(secondary_rgb)
    preview_titles.append(f"Image 2: {os.path.basename(image_path_2)}")

if preview_imgs:
    fig, axes = plt.subplots(1, len(preview_imgs), figsize=(5 * len(preview_imgs), 4))
    if len(preview_imgs) == 1:
        axes = [axes]
    for ax, img, t in zip(axes, preview_imgs, preview_titles):
        ax.imshow(img)
        ax.set_title(t, fontweight='bold', fontsize=11)
        ax.axis('off')
    plt.suptitle("Active Input Images Preview", fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()


---
## Section 4: Task 1 - Image Restoration and Segmentation
- **Noise Injection:** Gaussian AWGN ($\sigma=28.0$) & Salt-and-Pepper Impulse Noise (5%)
- **Spatial Filtering:** Gaussian Blur ($5\times 5, \sigma=1.5$) vs. Median Blur ($5\times 5$)
- **Quality Metrics:** Peak Signal-to-Noise Ratio (PSNR) & Structural Similarity Index (SSIM)
- **Segmentation:** Global Binary, Otsu's Optimal, and Adaptive Gaussian Thresholding
- **Morphological Operations:** Erosion, Dilation, Opening, and Closing ($5\times 5$ Rect Kernel)
- **Edge & Region Extraction:** Multi-stage Canny Edge Detection & Marker-controlled Watershed


In [ ]:
# Cell 4.1: Image Loading & Inspection
if primary_rgb is not None:
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.imshow(primary_rgb)
    ax.set_title(f"Original Input Image: {os.path.basename(image_path_1)}", fontsize=12, fontweight='bold')
    ax.set_xlabel(f"Width: {primary_rgb.shape[1]} px | Height: {primary_rgb.shape[0]} px | Channels: {primary_rgb.shape[2]}")
    save_path = os.path.join(TASK1_REST_DIR, "1_original_image.png")
    fig.savefig(save_path, bbox_inches='tight', dpi=150)
    plt.show()
    print(f"[✓] Saved original image reference to: {save_path}")
else:
    print("[Pending] Task 1.1 awaiting user image input in 'data/images/'.")


In [ ]:
# Cell 4.2: Gaussian Noise Synthesis
def add_gaussian_noise(image, mean=0.0, std=25.0, seed=RANDOM_SEED):
    """
    Adds synthetic Additive White Gaussian Noise (AWGN) to an image.
    Preserves data type (uint8) and clips values to the valid dynamic range [0, 255].
    """
    np.random.seed(seed)
    # Generate Gaussian distributed perturbations
    noise = np.random.normal(mean, std, image.shape)
    noisy_float = image.astype(np.float64) + noise
    # Constrain to valid 8-bit dynamic range and cast back
    noisy_uint8 = np.clip(noisy_float, 0, 255).astype(np.uint8)
    return noisy_uint8

if primary_rgb is not None:
    GAUSSIAN_STD = 28.0
    noisy_gaussian = add_gaussian_noise(primary_rgb, mean=0, std=GAUSSIAN_STD, seed=RANDOM_SEED)
    
    # Save noisy image
    save_path_gn = os.path.join(TASK1_REST_DIR, "2_gaussian_noisy_image.png")
    cv2.imwrite(save_path_gn, cv2.cvtColor(noisy_gaussian, cv2.COLOR_RGB2BGR))
    
    # Display comparison
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(primary_rgb)
    axes[0].set_title("Original Pristine Image", fontweight='bold')
    axes[0].axis('off')
    
    axes[1].imshow(noisy_gaussian)
    axes[1].set_title(r"Gaussian Noisy Image ($\sigma = " + str(GAUSSIAN_STD) + "$)", fontweight='bold')
    axes[1].axis('off')
    plt.tight_layout()
    fig.savefig(os.path.join(TASK1_REST_DIR, "2_gaussian_noise_comparison.png"), bbox_inches='tight', dpi=150)
    plt.show()
    print(f"[✓] Gaussian noise generated and saved to: {save_path_gn}")
else:
    print("[Pending] Task 1.2 awaiting user image input.")


In [ ]:
# Cell 4.3: Salt-and-Pepper (Impulse) Noise Synthesis
def add_salt_and_pepper_noise(image, density=0.04, salt_vs_pepper=0.5, seed=RANDOM_SEED):
    """
    Adds synthetic impulse (Salt-and-Pepper) noise to an image.
    - density: Fraction of total pixels corrupted.
    - salt_vs_pepper: Ratio of salt (white=255) vs pepper (black=0) noise.
    """
    np.random.seed(seed)
    noisy = image.copy()
    h, w = image.shape[:2]
    total_pixels = h * w
    num_noisy = int(density * total_pixels)
    
    # Generate random coordinates
    coords_y = np.random.randint(0, h, num_noisy)
    coords_x = np.random.randint(0, w, num_noisy)
    
    # Determine salt vs pepper allocation
    num_salt = int(num_noisy * salt_vs_pepper)
    
    # Salt noise (white = 255)
    noisy[coords_y[:num_salt], coords_x[:num_salt]] = 255
    # Pepper noise (black = 0)
    noisy[coords_y[num_salt:], coords_x[num_salt:]] = 0
    
    return noisy

if primary_rgb is not None:
    SP_DENSITY = 0.05
    noisy_sp = add_salt_and_pepper_noise(primary_rgb, density=SP_DENSITY, salt_vs_pepper=0.5, seed=RANDOM_SEED)
    
    # Save noisy image
    save_path_sp = os.path.join(TASK1_REST_DIR, "3_salt_pepper_noisy_image.png")
    cv2.imwrite(save_path_sp, cv2.cvtColor(noisy_sp, cv2.COLOR_RGB2BGR))
    
    # Display comparison
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(primary_rgb)
    axes[0].set_title("Original Pristine Image", fontweight='bold')
    axes[0].axis('off')
    
    axes[1].imshow(noisy_sp)
    axes[1].set_title(f"Salt & Pepper Noisy Image (Density = {SP_DENSITY*100:.1f}%)", fontweight='bold')
    axes[1].axis('off')
    plt.tight_layout()
    fig.savefig(os.path.join(TASK1_REST_DIR, "3_salt_pepper_noise_comparison.png"), bbox_inches='tight', dpi=150)
    plt.show()
    print(f"[✓] Salt-and-pepper noise generated and saved to: {save_path_sp}")
else:
    print("[Pending] Task 1.3 awaiting user image input.")


In [ ]:
# Cell 4.4: Spatial Filtering (Gaussian Blur vs. Median Blur)
# Filter parameters
KERNEL_SIZE = 5       # Must be a positive odd integer
GAUSSIAN_SIGMA = 1.5  # Standard deviation for Gaussian kernel

if primary_rgb is not None:
    # 1. Apply filters to Gaussian Noisy Image
    gauss_filtered_on_gauss = cv2.GaussianBlur(noisy_gaussian, (KERNEL_SIZE, KERNEL_SIZE), GAUSSIAN_SIGMA)
    median_filtered_on_gauss = cv2.medianBlur(noisy_gaussian, KERNEL_SIZE)
    
    # 2. Apply filters to Salt-and-Pepper Noisy Image
    gauss_filtered_on_sp = cv2.GaussianBlur(noisy_sp, (KERNEL_SIZE, KERNEL_SIZE), GAUSSIAN_SIGMA)
    median_filtered_on_sp = cv2.medianBlur(noisy_sp, KERNEL_SIZE)
    
    # Save restored images
    cv2.imwrite(os.path.join(TASK1_REST_DIR, "4_gauss_filt_on_gauss_noise.png"), cv2.cvtColor(gauss_filtered_on_gauss, cv2.COLOR_RGB2BGR))
    cv2.imwrite(os.path.join(TASK1_REST_DIR, "4_median_filt_on_gauss_noise.png"), cv2.cvtColor(median_filtered_on_gauss, cv2.COLOR_RGB2BGR))
    cv2.imwrite(os.path.join(TASK1_REST_DIR, "4_gauss_filt_on_sp_noise.png"), cv2.cvtColor(gauss_filtered_on_sp, cv2.COLOR_RGB2BGR))
    cv2.imwrite(os.path.join(TASK1_REST_DIR, "4_median_filt_on_sp_noise.png"), cv2.cvtColor(median_filtered_on_sp, cv2.COLOR_RGB2BGR))
    
    # Visualization Panel 1: Gaussian Noise Restoration
    fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
    axes[0].imshow(primary_rgb); axes[0].set_title("(A) Pristine Reference", fontweight='bold'); axes[0].axis('off')
    axes[1].imshow(noisy_gaussian); axes[1].set_title(r"(B) Gaussian Noisy ($\sigma=" + str(GAUSSIAN_STD) + "$)", fontweight='bold'); axes[1].axis('off')
    axes[2].imshow(gauss_filtered_on_gauss); axes[2].set_title(f"(C) Gaussian Filter ({KERNEL_SIZE}x{KERNEL_SIZE})", fontweight='bold'); axes[2].axis('off')
    axes[3].imshow(median_filtered_on_gauss); axes[3].set_title(f"(D) Median Filter ({KERNEL_SIZE}x{KERNEL_SIZE})", fontweight='bold'); axes[3].axis('off')
    plt.suptitle("Restoration Analysis: Gaussian Noise Treatment", fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    fig.savefig(os.path.join(TASK1_REST_DIR, "4_restoration_gaussian_noise_panel.png"), bbox_inches='tight', dpi=150)
    plt.show()
    
    # Visualization Panel 2: Salt & Pepper Noise Restoration
    fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
    axes[0].imshow(primary_rgb); axes[0].set_title("(A) Pristine Reference", fontweight='bold'); axes[0].axis('off')
    axes[1].imshow(noisy_sp); axes[1].set_title(f"(B) S&P Noisy ({SP_DENSITY*100:.0f}%)", fontweight='bold'); axes[1].axis('off')
    axes[2].imshow(gauss_filtered_on_sp); axes[2].set_title(f"(C) Gaussian Filter ({KERNEL_SIZE}x{KERNEL_SIZE})", fontweight='bold'); axes[2].axis('off')
    axes[3].imshow(median_filtered_on_sp); axes[3].set_title(f"(D) Median Filter ({KERNEL_SIZE}x{KERNEL_SIZE})", fontweight='bold'); axes[3].axis('off')
    plt.suptitle("Restoration Analysis: Salt-and-Pepper Impulse Noise Treatment", fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    fig.savefig(os.path.join(TASK1_REST_DIR, "4_restoration_sp_noise_panel.png"), bbox_inches='tight', dpi=150)
    plt.show()
    
    print("[✓] Spatial filtering completed and comparison panels saved.")
else:
    print("[Pending] Task 1.4 awaiting user image input.")


In [ ]:
# Cell 4.5: Quantitative Fidelity Evaluation: PSNR and SSIM
def calculate_psnr(reference, candidate):
    """
    Computes Peak Signal-to-Noise Ratio (PSNR) in decibels (dB).
    PSNR = 10 * log10(MAX_I^2 / MSE)
    """
    ref = reference.astype(np.float64)
    cand = candidate.astype(np.float64)
    mse = np.mean((ref - cand) ** 2)
    if mse == 0:
        return float('inf')
    max_pixel = 255.0
    psnr = 10.0 * np.log10((max_pixel ** 2) / mse)
    return round(psnr, 2)

def calculate_ssim(reference, candidate):
    """
    Computes Structural Similarity Index (SSIM) across color channels.
    Falls back to per-channel calculation if scikit-image is available.
    """
    if ski_metrics is not None:
        val = ski_metrics.structural_similarity(
            reference, candidate,
            channel_axis=2 if len(reference.shape) == 3 else None,
            data_range=255
        )
        return round(float(val), 4)
    else:
        # Fallback simplified SSIM calculation
        C1 = (0.01 * 255) ** 2
        C2 = (0.03 * 255) ** 2
        ref = reference.astype(np.float64)
        cand = candidate.astype(np.float64)
        mu1 = ndimage.uniform_filter(ref, size=11)
        mu2 = ndimage.uniform_filter(cand, size=11)
        sigma1_sq = ndimage.uniform_filter(ref**2, size=11) - mu1**2
        sigma2_sq = ndimage.uniform_filter(cand**2, size=11) - mu2**2
        sigma12 = ndimage.uniform_filter(ref * cand, size=11) - mu1 * mu2
        ssim_map = ((2 * mu1 * mu2 + C1) * (2 * sigma12 + C2)) / ((mu1**2 + mu2**2 + C1) * (sigma1_sq + sigma2_sq + C2))
        return round(float(np.mean(ssim_map)), 4)

if primary_rgb is not None:
    eval_cases = [
        ("Gaussian Noise", "Noisy Image (Unfiltered)", noisy_gaussian),
        ("Gaussian Noise", "Gaussian Blur (5x5, sigma=1.5)", gauss_filtered_on_gauss),
        ("Gaussian Noise", "Median Blur (5x5)", median_filtered_on_gauss),
        ("Salt & Pepper", "Noisy Image (Unfiltered)", noisy_sp),
        ("Salt & Pepper", "Gaussian Blur (5x5, sigma=1.5)", gauss_filtered_on_sp),
        ("Salt & Pepper", "Median Blur (5x5)", median_filtered_on_sp),
    ]
    
    restoration_metrics = []
    for noise_type, method, test_img in eval_cases:
        p_val = calculate_psnr(primary_rgb, test_img)
        s_val = calculate_ssim(primary_rgb, test_img)
        restoration_metrics.append({
            "Noise Type": noise_type,
            "Method / Filter": method,
            "PSNR (dB)": p_val,
            "SSIM": s_val
        })
    
    df_restoration = pd.DataFrame(restoration_metrics)
    csv_path_rest = os.path.join(METRICS_DIR, "task1_restoration_metrics.csv")
    df_restoration.to_csv(csv_path_rest, index=False)
    
    print("=" * 70)
    print("QUANTITATIVE RESTORATION METRICS (PSNR & SSIM):")
    display(df_restoration)
    print("=" * 70)
    print(f"[✓] Metrics persisted to: {csv_path_rest}")
else:
    print("[Pending] Task 1.5 awaiting user image input.")


In [ ]:
# Cell 4.6: Image Segmentation: Thresholding Techniques
if primary_rgb is not None:
    # Convert to single-channel 8-bit grayscale
    gray_primary = cv2.cvtColor(primary_rgb, cv2.COLOR_RGB2GRAY)
    
    # 1. Global Fixed Binary Thresholding (Fixed T = 127)
    T_FIXED = 127
    _, thresh_global = cv2.threshold(gray_primary, T_FIXED, 255, cv2.THRESH_BINARY)
    
    # 2. Otsu's Optimal Global Thresholding (Inter-class variance maximization)
    otsu_val, thresh_otsu = cv2.threshold(gray_primary, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    # 3. Adaptive Gaussian Thresholding (Local neighbourhood adaptation)
    thresh_adaptive = cv2.adaptiveThreshold(
        gray_primary, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        blockSize=11, C=2
    )
    
    # Persist thresholded outputs
    cv2.imwrite(os.path.join(TASK1_SEG_DIR, "1_thresh_global.png"), thresh_global)
    cv2.imwrite(os.path.join(TASK1_SEG_DIR, "1_thresh_otsu.png"), thresh_otsu)
    cv2.imwrite(os.path.join(TASK1_SEG_DIR, "1_thresh_adaptive.png"), thresh_adaptive)
    
    # Display comparison
    fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
    axes[0].imshow(gray_primary, cmap='gray'); axes[0].set_title("(A) Grayscale Input", fontweight='bold'); axes[0].axis('off')
    axes[1].imshow(thresh_global, cmap='gray'); axes[1].set_title(f"(B) Global Binary (T={T_FIXED})", fontweight='bold'); axes[1].axis('off')
    axes[2].imshow(thresh_otsu, cmap='gray'); axes[2].set_title(f"(C) Otsu's Binary (T*={otsu_val:.1f})", fontweight='bold'); axes[2].axis('off')
    axes[3].imshow(thresh_adaptive, cmap='gray'); axes[3].set_title("(D) Adaptive Gaussian", fontweight='bold'); axes[3].axis('off')
    plt.suptitle("Thresholding Techniques Comparison", fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    fig.savefig(os.path.join(TASK1_SEG_DIR, "1_thresholding_comparison.png"), bbox_inches='tight', dpi=150)
    plt.show()
    print(f"[✓] Thresholding experiments completed. Optimal Otsu threshold: {otsu_val:.2f}")
else:
    print("[Pending] Task 1.6 awaiting user image input.")


In [ ]:
# Cell 4.7: Mathematical Morphological Operations
if primary_rgb is not None:
    # Use Otsu's thresholded mask as the binary base
    base_binary = thresh_otsu.copy()
    
    # Define structuring element (Kernel)
    MORPH_KERNEL_SIZE = 5
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (MORPH_KERNEL_SIZE, MORPH_KERNEL_SIZE))
    
    # 1. Erosion: Shrinks foreground; strips isolated pixel noise
    morph_erosion = cv2.erode(base_binary, kernel, iterations=1)
    
    # 2. Dilation: Expands foreground; bridges small fractures and gaps
    morph_dilation = cv2.dilate(base_binary, kernel, iterations=1)
    
    # 3. Morphological Opening: Erosion followed by Dilation (cleans noise without altering shape)
    morph_opening = cv2.morphologyEx(base_binary, cv2.MORPH_OPEN, kernel, iterations=1)
    
    # 4. Morphological Closing: Dilation followed by Erosion (fills pinholes and connects segments)
    morph_closing = cv2.morphologyEx(base_binary, cv2.MORPH_CLOSE, kernel, iterations=1)
    
    # Save morphological artifacts
    cv2.imwrite(os.path.join(TASK1_SEG_DIR, "2_morph_erosion.png"), morph_erosion)
    cv2.imwrite(os.path.join(TASK1_SEG_DIR, "2_morph_dilation.png"), morph_dilation)
    cv2.imwrite(os.path.join(TASK1_SEG_DIR, "2_morph_opening.png"), morph_opening)
    cv2.imwrite(os.path.join(TASK1_SEG_DIR, "2_morph_closing.png"), morph_closing)
    
    # Visualization panel
    fig, axes = plt.subplots(1, 5, figsize=(20, 4))
    axes[0].imshow(base_binary, cmap='gray'); axes[0].set_title("(A) Binary Base", fontweight='bold'); axes[0].axis('off')
    axes[1].imshow(morph_erosion, cmap='gray'); axes[1].set_title("(B) Erosion", fontweight='bold'); axes[1].axis('off')
    axes[2].imshow(morph_dilation, cmap='gray'); axes[2].set_title("(C) Dilation", fontweight='bold'); axes[2].axis('off')
    axes[3].imshow(morph_opening, cmap='gray'); axes[3].set_title("(D) Opening", fontweight='bold'); axes[3].axis('off')
    axes[4].imshow(morph_closing, cmap='gray'); axes[4].set_title("(E) Closing", fontweight='bold'); axes[4].axis('off')
    plt.suptitle(f"Morphological Operations (Structuring Element: {MORPH_KERNEL_SIZE}x{MORPH_KERNEL_SIZE} Rect)", fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    fig.savefig(os.path.join(TASK1_SEG_DIR, "2_morphological_operations_panel.png"), bbox_inches='tight', dpi=150)
    plt.show()
    print("[✓] Morphological operations completed and saved.")
else:
    print("[Pending] Task 1.7 awaiting user image input.")


In [ ]:
# Cell 4.8: Canny Multi-Stage Edge Detection
if primary_rgb is not None:
    # Canny parameters
    CANNY_THRESH_LOW = 50
    CANNY_THRESH_HIGH = 150
    
    # Canny internally applies Gaussian smoothing, gradient computation, NMS, and hysteresis
    canny_edges = cv2.Canny(gray_primary, CANNY_THRESH_LOW, CANNY_THRESH_HIGH)
    
    # Persist edge map
    save_path_canny = os.path.join(TASK1_SEG_DIR, "3_canny_edges.png")
    cv2.imwrite(save_path_canny, canny_edges)
    
    # Visualization
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(primary_rgb)
    axes[0].set_title("Input RGB Image", fontweight='bold')
    axes[0].axis('off')
    
    axes[1].imshow(canny_edges, cmap='gray')
    axes[1].set_title(f"Canny Edge Map (T_low={CANNY_THRESH_LOW}, T_high={CANNY_THRESH_HIGH})", fontweight='bold')
    axes[1].axis('off')
    plt.tight_layout()
    fig.savefig(os.path.join(TASK1_SEG_DIR, "3_canny_edge_detection.png"), bbox_inches='tight', dpi=150)
    plt.show()
    print(f"[✓] Canny edge detection completed. Saved to: {save_path_canny}")
else:
    print("[Pending] Task 1.8 awaiting user image input.")


In [ ]:
# Cell 4.9: Marker-Controlled Watershed Segmentation
if primary_rgb is not None:
    # 1. Binarize using Otsu's thresholding
    _, binary_mask = cv2.threshold(gray_primary, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    
    # 2. Noise removal via morphological opening
    kernel_ws = np.ones((3, 3), np.uint8)
    opening_ws = cv2.morphologyEx(binary_mask, cv2.MORPH_OPEN, kernel_ws, iterations=2)
    
    # 3. Identify sure background area by dilating foreground
    sure_bg = cv2.dilate(opening_ws, kernel_ws, iterations=3)
    
    # 4. Identify sure foreground via Euclidean Distance Transform
    dist_transform = cv2.distanceTransform(opening_ws, cv2.DIST_L2, 5)
    ret_fg, sure_fg = cv2.threshold(dist_transform, 0.4 * dist_transform.max(), 255, 0)
    sure_fg = np.uint8(sure_fg)
    
    # 5. Unknown boundary region
    unknown_region = cv2.subtract(sure_bg, sure_fg)
    
    # 6. Marker labeling: Connected components
    num_markers, markers = cv2.connectedComponents(sure_fg)
    # Add 1 to all labels so that background is 1 instead of 0
    markers = markers + 1
    # Mark the unknown boundary region as 0
    markers[unknown_region == 255] = 0
    
    # 7. Apply Watershed algorithm (OpenCV requires BGR image)
    bgr_copy = cv2.cvtColor(primary_rgb, cv2.COLOR_RGB2BGR)
    markers_result = cv2.watershed(bgr_copy, markers)
    
    # 8. Overlay watershed boundaries on the RGB image
    watershed_overlay = primary_rgb.copy()
    # Boundaries are marked with -1
    watershed_overlay[markers_result == -1] = [255, 0, 0]  # Highlight boundaries in bold Red
    
    # Persist outputs
    cv2.imwrite(os.path.join(TASK1_SEG_DIR, "4_watershed_overlay.png"), cv2.cvtColor(watershed_overlay, cv2.COLOR_RGB2BGR))
    
    # Visualization panel
    fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
    axes[0].imshow(opening_ws, cmap='gray'); axes[0].set_title("(A) Morphological Opening", fontweight='bold'); axes[0].axis('off')
    axes[1].imshow(dist_transform, cmap='plasma'); axes[1].set_title("(B) Distance Transform", fontweight='bold'); axes[1].axis('off')
    axes[2].imshow(markers_result, cmap='tab20b'); axes[2].set_title(f"(C) Watershed Markers ({num_markers} segments)", fontweight='bold'); axes[2].axis('off')
    axes[3].imshow(watershed_overlay); axes[3].set_title("(D) Segmented Boundary Overlay", fontweight='bold'); axes[3].axis('off')
    plt.suptitle("Marker-Controlled Watershed Segmentation Pipeline", fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    fig.savefig(os.path.join(TASK1_SEG_DIR, "4_watershed_segmentation_panel.png"), bbox_inches='tight', dpi=150)
    plt.show()
    print(f"[✓] Watershed segmentation completed: {num_markers} component markers identified.")
else:
    print("[Pending] Task 1.9 awaiting user image input.")


---
## Section 5: Task 2 - Feature Extraction and Matching
- **Histogram of Oriented Gradients (HOG):** Gradient distributions across spatial cells
- **Scale-Invariant Feature Transform (SIFT):** Multi-scale DoG extrema & 128-D descriptors
- **Correspondence Matching:** Brute-Force Matcher with Lowe's Ratio Test ($d_1 < 0.75 \cdot d_2$)
- **Invariance Experiments:** Controlled Rotation ($45^\circ$), Scale ($0.5\times$), and Cross-Image Matching


In [ ]:
# Cell 5.1: Histogram of Oriented Gradients (HOG)
if primary_rgb is not None and hog is not None:
    # Convert image to single-channel grayscale
    gray_hog = cv2.cvtColor(primary_rgb, cv2.COLOR_RGB2GRAY)
    
    # HOG parameters
    HOG_ORIENTATIONS = 9
    HOG_PIXELS_PER_CELL = (8, 8)
    HOG_CELLS_PER_BLOCK = (2, 2)
    
    hog_descriptor, hog_image = hog(
        gray_hog,
        orientations=HOG_ORIENTATIONS,
        pixels_per_cell=HOG_PIXELS_PER_CELL,
        cells_per_block=HOG_CELLS_PER_BLOCK,
        visualize=True,
        block_norm='L2-Hys'
    )
    
    print("=" * 65)
    print("HOG DESCRIPTOR SPECIFICATIONS:")
    print(f"  Input Resolution        : {gray_hog.shape[1]} x {gray_hog.shape[0]}")
    print(f"  Orientation Bins        : {HOG_ORIENTATIONS}")
    print(f"  Pixels per Cell         : {HOG_PIXELS_PER_CELL}")
    print(f"  Cells per Block         : {HOG_CELLS_PER_BLOCK}")
    print(f"  Total Descriptor Length : {len(hog_descriptor):,} features")
    print("=" * 65)
    
    # Save HOG visualization
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].imshow(primary_rgb)
    axes[0].set_title("Input RGB Image", fontweight='bold')
    axes[0].axis('off')
    
    axes[1].imshow(hog_image, cmap='magma')
    axes[1].set_title(f"HOG Gradient Visualization ({len(hog_descriptor):,} dimensions)", fontweight='bold')
    axes[1].axis('off')
    plt.tight_layout()
    fig.savefig(os.path.join(TASK2_DIR, "1_hog_feature_visualization.png"), bbox_inches='tight', dpi=150)
    plt.show()
    print("[✓] HOG extraction complete and visualization saved.")
else:
    print("[Pending / Notice] Task 2.1 awaiting image input or scikit-image installation.")


In [ ]:
# Cell 5.2: SIFT Keypoint Detection and Description
if primary_rgb is not None and cv2 is not None:
    # Initialize SIFT detector
    try:
        sift = cv2.SIFT_create()
        sift_available = True
    except AttributeError:
        sift_available = False
        print("[!] SIFT is unavailable in this OpenCV build. Ensure 'opencv-python' is installed.")

    if sift_available:
        gray_sift = cv2.cvtColor(primary_rgb, cv2.COLOR_RGB2GRAY)
        keypoints, descriptors = sift.detectAndCompute(gray_sift, None)
        
        # Render keypoints with scale circles and orientation vectors
        sift_visual = cv2.drawKeypoints(
            primary_rgb, keypoints, None,
            flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
        )
        
        print("=" * 65)
        print("SIFT DETECTOR & DESCRIPTOR REPORT:")
        print(f"  Keypoints Detected    : {len(keypoints):,}")
        if descriptors is not None:
            print(f"  Descriptor Shape      : {descriptors.shape} (N_points x 128 dimensions)")
            print(f"  Feature DataType      : {descriptors.dtype}")
        print("=" * 65)
        
        # Persist and display
        fig, axes = plt.subplots(1, 2, figsize=(15, 6))
        axes[0].imshow(primary_rgb)
        axes[0].set_title("Input Image", fontweight='bold')
        axes[0].axis('off')
        
        axes[1].imshow(sift_visual)
        axes[1].set_title(f"SIFT Rich Keypoints ({len(keypoints)} detected)", fontweight='bold')
        axes[1].axis('off')
        plt.tight_layout()
        fig.savefig(os.path.join(TASK2_DIR, "2_sift_keypoints.png"), bbox_inches='tight', dpi=150)
        plt.show()
        print("[✓] SIFT keypoints detected and visualized.")
else:
    print("[Pending] Task 2.2 awaiting user image input.")


In [ ]:
# Cell 5.3: Feature Matching Between Two Supplied Images
def match_sift_features(img1_rgb, img2_rgb, ratio_thresh=0.75):
    """
    Detects SIFT features in two images and matches them using Brute-Force Matcher
    with Lowe's Second-Nearest-Neighbor Ratio Test.
    """
    sift = cv2.SIFT_create()
    gray1 = cv2.cvtColor(img1_rgb, cv2.COLOR_RGB2GRAY)
    gray2 = cv2.cvtColor(img2_rgb, cv2.COLOR_RGB2GRAY)
    
    kp1, des1 = sift.detectAndCompute(gray1, None)
    kp2, des2 = sift.detectAndCompute(gray2, None)
    
    if des1 is None or des2 is None or len(kp1) == 0 or len(kp2) == 0:
        return kp1, kp2, [], None
    
    # Brute-Force Matcher with L2 distance
    bf = cv2.BFMatcher(cv2.NORM_L2)
    raw_matches = bf.knnMatch(des1, des2, k=2)
    
    # Apply Lowe's Ratio Test: d1 / d2 < ratio_thresh
    good_matches = []
    for pair in raw_matches:
        if len(pair) == 2:
            m, n = pair
            if m.distance < ratio_thresh * n.distance:
                good_matches.append(m)
                
    # Sort good matches by descriptor distance
    good_matches = sorted(good_matches, key=lambda x: x.distance)
    
    # Draw match correspondences
    match_img = cv2.drawMatches(
        img1_rgb, kp1,
        img2_rgb, kp2,
        good_matches[:60], None, # Display top 60 good matches for visual clarity
        flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
    )
    
    stats = {
        'kp1_count': len(kp1),
        'kp2_count': len(kp2),
        'raw_matches': len(raw_matches),
        'good_matches': len(good_matches),
        'retention_pct': round((len(good_matches) / len(raw_matches) * 100), 2) if len(raw_matches) > 0 else 0.0
    }
    return kp1, kp2, good_matches, match_img, stats

if primary_rgb is not None and secondary_rgb is not None:
    kp1, kp2, good_matches, match_vis, stats_2images = match_sift_features(primary_rgb, secondary_rgb, ratio_thresh=0.75)
    
    print("=" * 65)
    print("TWO-IMAGE SIFT MATCHING RESULTS:")
    print(f"  Image 1 Keypoints     : {stats_2images['kp1_count']}")
    print(f"  Image 2 Keypoints     : {stats_2images['kp2_count']}")
    print(f"  Raw KNN (k=2) Matches : {stats_2images['raw_matches']}")
    print(f"  Lowe's Retained Matches: {stats_2images['good_matches']}")
    print(f"  Match Retention Rate  : {stats_2images['retention_pct']}%")
    print("=" * 65)
    
    if match_vis is not None:
        fig, ax = plt.subplots(figsize=(16, 7))
        ax.imshow(match_vis)
        ax.set_title(f"SIFT Feature Correspondence: {os.path.basename(image_path_1)} <--> {os.path.basename(image_path_2)} ({stats_2images['good_matches']} matches)", fontweight='bold')
        ax.axis('off')
        fig.savefig(os.path.join(TASK2_DIR, "3_sift_two_image_matches.png"), bbox_inches='tight', dpi=150)
        plt.show()
else:
    print("[Pending / Notice] Task 5.3 requires two images in 'data/images/' to demonstrate cross-image correspondence.")


In [ ]:
# Cell 5.4: Geometric Invariance Experiments (Rotation & Resizing)
if primary_rgb is not None:
    h, w = primary_rgb.shape[:2]
    
    # 1. Transformation 1: Controlled Rotation (45 degrees)
    ROT_ANGLE = 45.0
    rot_center = (w // 2, h // 2)
    M_rot = cv2.getRotationMatrix2D(rot_center, ROT_ANGLE, 1.0)
    img_rotated = cv2.warpAffine(primary_rgb, M_rot, (w, h), borderMode=cv2.BORDER_REFLECT)
    
    # 2. Transformation 2: Controlled Resizing / Scale Reduction (0.5x)
    SCALE_FACTOR = 0.5
    img_scaled = cv2.resize(primary_rgb, (int(w * SCALE_FACTOR), int(h * SCALE_FACTOR)), interpolation=cv2.INTER_AREA)
    
    # Execute matching experiments
    _, _, _, match_rot_vis, stats_rot = match_sift_features(primary_rgb, img_rotated, ratio_thresh=0.75)
    _, _, _, match_scale_vis, stats_scale = match_sift_features(primary_rgb, img_scaled, ratio_thresh=0.75)
    
    # Visualize Rotation Invariance
    fig, ax = plt.subplots(figsize=(15, 6))
    ax.imshow(match_rot_vis)
    ax.set_title(f"SIFT Invariance Experiment 1: Original vs. Rotated ({ROT_ANGLE}°) - Retained Matches: {stats_rot['good_matches']}", fontweight='bold')
    ax.axis('off')
    fig.savefig(os.path.join(TASK2_DIR, "4_invariance_rotation.png"), bbox_inches='tight', dpi=150)
    plt.show()
    
    # Visualize Scale Invariance
    fig, ax = plt.subplots(figsize=(15, 6))
    ax.imshow(match_scale_vis)
    ax.set_title(f"SIFT Invariance Experiment 2: Original vs. Scaled ({SCALE_FACTOR}x) - Retained Matches: {stats_scale['good_matches']}", fontweight='bold')
    ax.axis('off')
    fig.savefig(os.path.join(TASK2_DIR, "5_invariance_scale.png"), bbox_inches='tight', dpi=150)
    plt.show()
    
    # Compile Invariance Summary Table
    invariance_data = [
        {
            "Experiment": "Exp 1: Rotation Invariance",
            "Condition": f"In-plane Rotation ({ROT_ANGLE}°)",
            "Ref Keypoints": stats_rot['kp1_count'],
            "Target Keypoints": stats_rot['kp2_count'],
            "Raw Matches": stats_rot['raw_matches'],
            "Good Matches (Lowe)": stats_rot['good_matches'],
            "Retention Rate (%)": stats_rot['retention_pct']
        },
        {
            "Experiment": "Exp 2: Scale Invariance",
            "Condition": f"Resized Down ({SCALE_FACTOR}x)",
            "Ref Keypoints": stats_scale['kp1_count'],
            "Target Keypoints": stats_scale['kp2_count'],
            "Raw Matches": stats_scale['raw_matches'],
            "Good Matches (Lowe)": stats_scale['good_matches'],
            "Retention Rate (%)": stats_scale['retention_pct']
        }
    ]
    
    if secondary_rgb is not None and 'stats_2images' in locals():
        invariance_data.append({
            "Experiment": "Exp 3: Cross-Image Matching",
            "Condition": f"Different View / Scene ({os.path.basename(image_path_2)})",
            "Ref Keypoints": stats_2images['kp1_count'],
            "Target Keypoints": stats_2images['kp2_count'],
            "Raw Matches": stats_2images['raw_matches'],
            "Good Matches (Lowe)": stats_2images['good_matches'],
            "Retention Rate (%)": stats_2images['retention_pct']
        })
    
    df_features = pd.DataFrame(invariance_data)
    csv_feat_path = os.path.join(METRICS_DIR, "task2_feature_metrics.csv")
    df_features.to_csv(csv_feat_path, index=False)
    
    print("=" * 75)
    print("TASK 2 FEATURE MATCHING & INVARIANCE EXPERIMENTAL METRICS:")
    display(df_features)
    print("=" * 75)
    print(f"[✓] Feature metrics exported to: {csv_feat_path}")
else:
    print("[Pending] Task 2.4 awaiting user image input.")


---
## Section 6: Task 3 - Object Recognition
- **Classical Detection:** Viola-Jones Haar Feature Cascade (Frontal Face)
- **Deep Learning Detection:** Ultralytics YOLOv8 Single-Stage Convolutional Detector
- **Comparative Evaluation:** Latency, target specificity, and detection bounding boxes


In [ ]:
# Cell 6.1: Classical Object Detection: Haar Feature Cascade
haar_detections_count = 0
haar_time_ms = 0.0

if primary_rgb is not None and cv2 is not None:
    if not hasattr(cv2, 'CascadeClassifier'):
        print("[!] Notice: 'cv2.CascadeClassifier' is not present in this OpenCV build (e.g. OpenCV 5.0.0 alpha).")
        print("    To enable classical Haar Cascade detection, run: !pip install 'opencv-python<5.0.0.0'")
    else:
        # Locate OpenCV's packaged Haar cascade for frontal face detection
        cascade_filename = "haarcascade_frontalface_default.xml"
        cascade_path = os.path.join(cv2.data.haarcascades, cascade_filename) if hasattr(cv2, 'data') else cascade_filename
        
        if os.path.exists(cascade_path):
            face_cascade = cv2.CascadeClassifier(cascade_path)
            gray_haar = cv2.cvtColor(primary_rgb, cv2.COLOR_RGB2GRAY)
            
            # Benchmark inference time (excluding file loading)
            t_start = time.perf_counter()
            detected_faces = face_cascade.detectMultiScale(
                gray_haar,
                scaleFactor=1.1,
                minNeighbors=5,
                minSize=(30, 30)
            )
            haar_time_ms = round((time.perf_counter() - t_start) * 1000, 2)
            haar_detections_count = len(detected_faces)
            
            # Annotate image
            annotated_haar = primary_rgb.copy()
            for (x, y, w_box, h_box) in detected_faces:
                cv2.rectangle(annotated_haar, (x, y), (x + w_box, y + h_box), (0, 255, 0), 3)
                cv2.putText(
                    annotated_haar, "Face", (x, max(y - 10, 15)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2
                )
                
            # Save annotated detection image
            save_path_haar = os.path.join(TASK3_DIR, "1_haarcascade_detections.png")
            cv2.imwrite(save_path_haar, cv2.cvtColor(annotated_haar, cv2.COLOR_RGB2BGR))
            
            # Display visualization
            fig, ax = plt.subplots(figsize=(8, 6))
            ax.imshow(annotated_haar)
            ax.set_title(f"Haar Cascade Detection (Faces Found: {haar_detections_count} | Latency: {haar_time_ms:.1f} ms)", fontweight='bold')
            ax.axis('off')
            plt.tight_layout()
            fig.savefig(os.path.join(TASK3_DIR, "1_haarcascade_display.png"), bbox_inches='tight', dpi=150)
            plt.show()
            
            print("=" * 65)
            print("HAAR CASCADE DETECTION REPORT:")
            print(f"  Classifier XML        : {cascade_filename}")
            print(f"  Faces Detected        : {haar_detections_count}")
            print(f"  Measured Latency      : {haar_time_ms} ms")
            print("=" * 65)
        else:
            print(f"[!] Haar cascade file not found at: {cascade_path}")
else:
    print("[Pending] Task 6.1 awaiting user image input.")


In [ ]:
# Cell 6.2: Modern Deep Learning Object Detection: Ultralytics YOLO
yolo_detections_count = 0
yolo_time_ms = 0.0
yolo_records = []

if primary_rgb is not None:
    if YOLO is not None:
        try:
            print("[*] Initializing Ultralytics YOLOv8n (Pre-trained Nano Weights)...")
            yolo_model = YOLO("yolov8n.pt")
            
            # Run inference
            CONFIDENCE_THRESHOLD = 0.25
            t_start = time.perf_counter()
            results = yolo_model.predict(
                source=primary_rgb,
                conf=CONFIDENCE_THRESHOLD,
                verbose=False
            )
            yolo_time_ms = round((time.perf_counter() - t_start) * 1000, 2)
            
            annotated_yolo = primary_rgb.copy()
            if len(results) > 0 and results[0].boxes is not None:
                boxes = results[0].boxes
                yolo_detections_count = len(boxes)
                
                for box in boxes:
                    xyxy = box.xyxy[0].cpu().numpy().astype(int)
                    conf = float(box.conf[0].cpu().numpy())
                    cls_id = int(box.cls[0].cpu().numpy())
                    cls_name = yolo_model.names[cls_id]
                    
                    yolo_records.append({
                        "Class ID": cls_id,
                        "Class Name": cls_name,
                        "Confidence": round(conf, 4),
                        "x_min": xyxy[0], "y_min": xyxy[1],
                        "x_max": xyxy[2], "y_max": xyxy[3]
                    })
                    
                    # Draw bounding box & label
                    cv2.rectangle(annotated_yolo, (xyxy[0], xyxy[1]), (xyxy[2], xyxy[3]), (255, 120, 0), 3)
                    label = f"{cls_name} {conf:.2f}"
                    cv2.putText(
                        annotated_yolo, label, (xyxy[0], max(xyxy[1] - 8, 15)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 120, 0), 2
                    )
            
            # Save annotated image and detection CSV
            save_path_yolo = os.path.join(TASK3_DIR, "2_yolo_detections.png")
            cv2.imwrite(save_path_yolo, cv2.cvtColor(annotated_yolo, cv2.COLOR_RGB2BGR))
            
            df_yolo_records = pd.DataFrame(yolo_records)
            csv_yolo_records = os.path.join(TASK3_DIR, "yolo_detection_records.csv")
            df_yolo_records.to_csv(csv_yolo_records, index=False)
            
            # Display visualization
            fig, ax = plt.subplots(figsize=(9, 6.5))
            ax.imshow(annotated_yolo)
            ax.set_title(f"YOLOv8 Detection (Objects Found: {yolo_detections_count} | Latency: {yolo_time_ms:.1f} ms)", fontweight='bold')
            ax.axis('off')
            plt.tight_layout()
            fig.savefig(os.path.join(TASK3_DIR, "2_yolo_display.png"), bbox_inches='tight', dpi=150)
            plt.show()
            
            print("=" * 65)
            print("ULTRALYTICS YOLO INFERENCE REPORT:")
            print(f"  Model Architecture    : YOLOv8n (COCO 80 classes)")
            print(f"  Objects Detected      : {yolo_detections_count}")
            print(f"  Inference Latency     : {yolo_time_ms} ms")
            if len(yolo_records) > 0:
                print("\nDetected Entities:")
                for r in yolo_records:
                    print(f"    - {r['Class Name']} (Confidence: {r['Confidence']:.2f}) at [{r['x_min']}, {r['y_min']}, {r['x_max']}, {r['y_max']}]")
            print("=" * 65)
            
        except Exception as e:
            print(f"[!] YOLO inference error or network limitation: {e}")
            print("    Ultralytics needs network access to download weights once, or a local 'yolov8n.pt' file.")
    else:
        print("[Notice] Ultralytics is not installed. Install via `pip install ultralytics`.")
else:
    print("[Pending] Task 6.2 awaiting user image input.")


In [ ]:
# Cell 6.3: Comparative Synthesis: Classical Haar Cascade vs. Deep Learning YOLO
if primary_rgb is not None:
    comparison_records = [
        {
            "Method": "Haar Feature Cascade (Viola-Jones)",
            "Paradigm": "Classical Handcrafted Features + AdaBoost",
            "Detection Target": "Domain-Specific (Human Faces)",
            "Detections Found": haar_detections_count,
            "Latency (ms)": haar_time_ms,
            "Hardware Target": "Ultra-lightweight on CPU",
            "Key Advantages": "Zero deep learning dependencies; runs in milliseconds on low-power microcontrollers.",
            "Inherent Limitations": "Rigid geometric templates; prone to false positives on complex backgrounds; fails on non-frontal angles."
        },
        {
            "Method": "YOLOv8 Single-Stage Detector",
            "Paradigm": "Deep Convolutional Neural Network (End-to-End)",
            "Detection Target": "General Multi-Class (80 COCO Categories)",
            "Detections Found": yolo_detections_count,
            "Latency (ms)": yolo_time_ms,
            "Hardware Target": "CPU / GPU Accelerated",
            "Key Advantages": "High semantic richness; contextual global reasoning; invariant to varied poses, scales, and partial occlusions.",
            "Inherent Limitations": "Requires heavier computational resources and pre-trained neural network weights."
        }
    ]
    
    df_task3_comp = pd.DataFrame(comparison_records)
    csv_comp_task3 = os.path.join(METRICS_DIR, "task3_detection_metrics.csv")
    df_task3_comp.to_csv(csv_comp_task3, index=False)
    
    print("=" * 75)
    print("TASK 3 OBJECT RECOGNITION COMPARATIVE MATRIX:")
    display(df_task3_comp)
    print("=" * 75)
    print(f"[✓] Detection benchmark persisted to: {csv_comp_task3}")
else:
    print("[Pending] Task 6.3 awaiting user image input.")


---
## Section 7: Task 4 - Video Motion Estimation
- **Video Inspection:** Sequential stream decoding and frame sampling
- **Method 1:** Temporal Consecutive Frame Differencing ($|I_t - I_{t-1}| > T$)
- **Method 2:** Gaussian Mixture Model Background Subtraction (MOG2) with Shadow Detection
- **Method 3:** Sparse Lucas-Kanade Pyramidal Optical Flow on Shi-Tomasi Corners
- **Throughput Benchmark:** Measured FPS, sensitivity, and computational complexity


In [ ]:
# Cell 7.1: Video Stream Inspection & Sample Frame Extraction
if video_meta is not None and cv2 is not None:
    cap = cv2.VideoCapture(video_meta['filepath'])
    sample_frames = []
    
    # Read first few frames
    for i in range(min(4, video_meta['frames'] if video_meta['frames'] > 0 else 4)):
        ret, frame = cap.read()
        if ret:
            sample_frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        else:
            break
    cap.release()
    
    if len(sample_frames) > 0:
        fig, axes = plt.subplots(1, len(sample_frames), figsize=(16, 4))
        if len(sample_frames) == 1:
            axes = [axes]
        for idx, f in enumerate(sample_frames):
            axes[idx].imshow(f)
            axes[idx].set_title(f"Sample Frame #{idx+1}", fontweight='bold')
            axes[idx].axis('off')
        plt.suptitle(f"Video Stream Inspection: '{video_meta['filename']}' ({video_meta['fps']:.1f} FPS)", fontsize=13, fontweight='bold', y=1.02)
        plt.tight_layout()
        fig.savefig(os.path.join(TASK4_DIR, "0_video_inspection_frames.png"), bbox_inches='tight', dpi=150)
        plt.show()
        print(f"[✓] Successfully sampled {len(sample_frames)} frames from: {video_meta['filename']}")
    else:
        print("[!] Unable to read initial frames from video file.")
else:
    print("[Pending] Task 7.1 awaiting video file in 'data/videos/'.")


In [ ]:
# Cell 7.2: Motion Detection Method 1: Temporal Frame Differencing
DIFF_THRESHOLD = 25
FRAME_LIMIT = 90  # Maximum frames processed to maintain high execution speed

fd_total_time = 0.0
fd_processed_frames = 0
representative_fd = None

if video_meta is not None and cv2 is not None:
    cap = cv2.VideoCapture(video_meta['filepath'])
    ret, prev_frame = cap.read()
    
    if ret:
        h_vid, w_vid = prev_frame.shape[:2]
        prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)
        prev_gray = cv2.GaussianBlur(prev_gray, (5, 5), 0)
        
        # Initialize VideoWriter to produce annotated output MP4 video
        out_vid_path_fd = os.path.join(TASK4_DIR, "1_frame_differencing_annotated.mp4")
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out_writer_fd = cv2.VideoWriter(out_vid_path_fd, fourcc, video_meta['fps'], (w_vid, h_vid))
        
        kernel_fd = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
        t_start = time.perf_counter()
        
        frame_idx = 1
        progression_frames = []
        
        while frame_idx < FRAME_LIMIT:
            ret, curr_frame = cap.read()
            if not ret:
                break
                
            curr_gray = cv2.cvtColor(curr_frame, cv2.COLOR_BGR2GRAY)
            curr_gray = cv2.GaussianBlur(curr_gray, (5, 5), 0)
            
            # Compute absolute pixel-wise difference: |I_t - I_{t-1}|
            diff = cv2.absdiff(curr_gray, prev_gray)
            
            # Threshold difference map
            _, motion_mask = cv2.threshold(diff, DIFF_THRESHOLD, 255, cv2.THRESH_BINARY)
            
            # Clean noise using morphological opening
            motion_mask_clean = cv2.morphologyEx(motion_mask, cv2.MORPH_OPEN, kernel_fd)
            
            # Create color overlay for annotated video: highlight moving pixels in green
            annotated_frame = curr_frame.copy()
            annotated_frame[motion_mask_clean == 255] = [0, 255, 0]
            cv2.putText(annotated_frame, f"Frame {frame_idx} | Differencing", (30, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
            out_writer_fd.write(annotated_frame)
            
            # Collect sample frames at intervals for multi-step visual display
            if frame_idx in [15, 30, 45, 60]:
                progression_frames.append({
                    'frame': cv2.cvtColor(curr_frame, cv2.COLOR_BGR2RGB),
                    'mask': motion_mask_clean,
                    'idx': frame_idx
                })
                
            if frame_idx == min(25, FRAME_LIMIT // 2):
                representative_fd = {
                    'frame_rgb': cv2.cvtColor(curr_frame, cv2.COLOR_BGR2RGB),
                    'diff': diff,
                    'mask': motion_mask_clean,
                    'frame_num': frame_idx
                }
                
            prev_gray = curr_gray
            frame_idx += 1
            
        fd_total_time = time.perf_counter() - t_start
        fd_processed_frames = frame_idx
        cap.release()
        out_writer_fd.release()
        
        fd_fps = round(fd_processed_frames / fd_total_time, 1) if fd_total_time > 0 else 0
        print("=" * 65)
        print("TEMPORAL FRAME DIFFERENCING SUMMARY:")
        print(f"  Frames Analyzed       : {fd_processed_frames}")
        print(f"  Processing Time       : {fd_total_time:.3f} s")
        print(f"  Throughput Speed      : {fd_fps} FPS")
        print(f"  Annotated Video Saved : {out_vid_path_fd}")
        print("=" * 65)
        
        if representative_fd is not None:
            fig, axes = plt.subplots(1, 3, figsize=(16, 5))
            axes[0].imshow(representative_fd['frame_rgb'])
            axes[0].set_title(f"(A) Frame #{representative_fd['frame_num']}", fontweight='bold'); axes[0].axis('off')
            
            axes[1].imshow(representative_fd['diff'], cmap='inferno')
            axes[1].set_title("(B) Raw Absolute Difference Map", fontweight='bold'); axes[1].axis('off')
            
            axes[2].imshow(representative_fd['mask'], cmap='gray')
            axes[2].set_title(f"(C) Thresholded Motion Mask (T={DIFF_THRESHOLD})", fontweight='bold'); axes[2].axis('off')
            plt.suptitle("Method 1: Temporal Frame Differencing Motion Estimation", fontsize=13, fontweight='bold', y=1.02)
            plt.tight_layout()
            fig.savefig(os.path.join(TASK4_DIR, "1_frame_differencing.png"), bbox_inches='tight', dpi=150)
            plt.show()
    else:
        cap.release()
        print("[!] Unable to initialize frame differencing on video stream.")
else:
    print("[!] Task 7.2 Pending: No video file found.")
    print("    Upload a video (.mp4, .avi) to 'data/videos/' and re-run this cell to generate motion estimation outputs.")


In [ ]:
# Cell 7.3: Motion Detection Method 2: Gaussian Mixture Model Background Subtraction (MOG2)
mog_total_time = 0.0
mog_processed_frames = 0
representative_mog = None

if video_meta is not None and cv2 is not None:
    cap = cv2.VideoCapture(video_meta['filepath'])
    ret, first_frame = cap.read()
    
    if ret:
        h_vid, w_vid = first_frame.shape[:2]
        subtractor = cv2.createBackgroundSubtractorMOG2(history=300, varThreshold=25, detectShadows=True)
        kernel_mog = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        
        out_vid_path_mog = os.path.join(TASK4_DIR, "2_mog2_background_subtraction_annotated.mp4")
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out_writer_mog = cv2.VideoWriter(out_vid_path_mog, fourcc, video_meta['fps'], (w_vid, h_vid))
        
        t_start = time.perf_counter()
        frame_idx = 0
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
        
        while frame_idx < FRAME_LIMIT:
            ret, frame = cap.read()
            if not ret:
                break
                
            # Apply MOG2: foreground=255, shadows=127, background=0
            fg_mask_raw = subtractor.apply(frame)
            
            # Suppress shadows: keep only true moving foreground
            _, fg_mask_binary = cv2.threshold(fg_mask_raw, 250, 255, cv2.THRESH_BINARY)
            fg_mask_clean = cv2.morphologyEx(fg_mask_binary, cv2.MORPH_OPEN, kernel_mog)
            
            # Find contours and draw bounding boxes around moving entities
            contours, _ = cv2.findContours(fg_mask_clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            annotated_frame = frame.copy()
            for cnt in contours:
                if cv2.contourArea(cnt) > 250:
                    x, y, w_cnt, h_cnt = cv2.boundingRect(cnt)
                    cv2.rectangle(annotated_frame, (x, y), (x + w_cnt, y + h_cnt), (0, 255, 0), 2)
                    cv2.putText(annotated_frame, "Motion", (x, max(y - 8, 15)),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
                                
            cv2.putText(annotated_frame, f"Frame {frame_idx} | MOG2 GMM", (30, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
            out_writer_mog.write(annotated_frame)
            
            if frame_idx == min(25, FRAME_LIMIT // 2):
                representative_mog = {
                    'frame_rgb': cv2.cvtColor(frame, cv2.COLOR_BGR2RGB),
                    'annotated_rgb': cv2.cvtColor(annotated_frame, cv2.COLOR_BGR2RGB),
                    'raw_mask': fg_mask_raw,
                    'clean_mask': fg_mask_clean,
                    'frame_num': frame_idx
                }
                
            frame_idx += 1
            
        mog_total_time = time.perf_counter() - t_start
        mog_processed_frames = frame_idx
        cap.release()
        out_writer_mog.release()
        
        mog_fps = round(mog_processed_frames / mog_total_time, 1) if mog_total_time > 0 else 0
        print("=" * 65)
        print("MOG2 BACKGROUND SUBTRACTION SUMMARY:")
        print(f"  Frames Analyzed       : {mog_processed_frames}")
        print(f"  Processing Time       : {mog_total_time:.3f} s")
        print(f"  Throughput Speed      : {mog_fps} FPS")
        print(f"  Annotated Video Saved : {out_vid_path_mog}")
        print("=" * 65)
        
        if representative_mog is not None:
            fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
            axes[0].imshow(representative_mog['frame_rgb']); axes[0].set_title(f"(A) Original Frame #{representative_mog['frame_num']}", fontweight='bold'); axes[0].axis('off')
            axes[1].imshow(representative_mog['raw_mask'], cmap='gray'); axes[1].set_title("(B) MOG2 Mask (Shadows=127)", fontweight='bold'); axes[1].axis('off')
            axes[2].imshow(representative_mog['clean_mask'], cmap='gray'); axes[2].set_title("(C) Cleaned Foreground Mask", fontweight='bold'); axes[2].axis('off')
            axes[3].imshow(representative_mog['annotated_rgb']); axes[3].set_title("(D) Motion Bounding Boxes", fontweight='bold'); axes[3].axis('off')
            plt.suptitle("Method 2: MOG2 Gaussian Mixture Model Background Subtraction", fontsize=13, fontweight='bold', y=1.02)
            plt.tight_layout()
            fig.savefig(os.path.join(TASK4_DIR, "2_mog2_background_subtraction.png"), bbox_inches='tight', dpi=150)
            plt.show()
    else:
        cap.release()
        print("[!] Unable to read first frame from video.")
else:
    print("[!] Task 7.3 Pending: No video file found.")
    print("    Upload a video (.mp4, .avi) to 'data/videos/' and re-run this cell to generate MOG2 outputs.")


In [ ]:
# Cell 7.4: Motion Detection Method 3: Sparse Lucas-Kanade Optical Flow
lk_total_time = 0.0
lk_processed_frames = 0
representative_lk = None

if video_meta is not None and cv2 is not None:
    cap = cv2.VideoCapture(video_meta['filepath'])
    ret, old_frame = cap.read()
    
    if ret:
        h_vid, w_vid = old_frame.shape[:2]
        old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)
        
        # Initialize VideoWriter for optical flow vector trajectories
        out_vid_path_lk = os.path.join(TASK4_DIR, "3_lucas_kanade_optical_flow_annotated.mp4")
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out_writer_lk = cv2.VideoWriter(out_vid_path_lk, fourcc, video_meta['fps'], (w_vid, h_vid))
        
        feature_params = dict(maxCorners=120, qualityLevel=0.3, minDistance=7, blockSize=7)
        lk_params = dict(winSize=(15, 15), maxLevel=2, criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03))
        
        p0 = cv2.goodFeaturesToTrack(old_gray, mask=None, **feature_params)
        mask_track = np.zeros_like(old_frame)
        color_lk = np.random.randint(0, 255, (150, 3))
        
        t_start = time.perf_counter()
        frame_idx = 1
        
        while frame_idx < FRAME_LIMIT:
            ret, frame = cap.read()
            if not ret:
                break
                
            frame_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            
            if p0 is not None and len(p0) > 0:
                p1, st, err = cv2.calcOpticalFlowPyrLK(old_gray, frame_gray, p0, None, **lk_params)
                
                if p1 is not None and st is not None:
                    good_new = p1[st == 1]
                    good_old = p0[st == 1]
                    
                    for i, (new, old) in enumerate(zip(good_new, good_old)):
                        a, b = new.ravel()
                        c, d = old.ravel()
                        mask_track = cv2.line(mask_track, (int(a), int(b)), (int(c), int(d)), color_lk[i % len(color_lk)].tolist(), 2)
                        frame = cv2.circle(frame, (int(a), int(b)), 4, color_lk[i % len(color_lk)].tolist(), -1)
                        
                    if len(good_new) < 15:
                        p0 = cv2.goodFeaturesToTrack(frame_gray, mask=None, **feature_params)
                    else:
                        p0 = good_new.reshape(-1, 1, 2)
            else:
                p0 = cv2.goodFeaturesToTrack(frame_gray, mask=None, **feature_params)
                
            output_flow_frame = cv2.add(frame, mask_track)
            cv2.putText(output_flow_frame, f"Frame {frame_idx} | Lucas-Kanade Flow", (30, 40),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
            out_writer_lk.write(output_flow_frame)
            
            if frame_idx == min(35, FRAME_LIMIT // 2):
                representative_lk = {
                    'flow_rgb': cv2.cvtColor(output_flow_frame, cv2.COLOR_BGR2RGB),
                    'frame_num': frame_idx
                }
                
            old_gray = frame_gray.copy()
            frame_idx += 1
            
        lk_total_time = time.perf_counter() - t_start
        lk_processed_frames = frame_idx
        cap.release()
        out_writer_lk.release()
        
        lk_fps = round(lk_processed_frames / lk_total_time, 1) if lk_total_time > 0 else 0
        print("=" * 65)
        print("LUCAS-KANADE OPTICAL FLOW SUMMARY:")
        print(f"  Frames Analyzed       : {lk_processed_frames}")
        print(f"  Processing Time       : {lk_total_time:.3f} s")
        print(f"  Throughput Speed      : {lk_fps} FPS")
        print(f"  Annotated Video Saved : {out_vid_path_lk}")
        print("=" * 65)
        
        if representative_lk is not None:
            fig, ax = plt.subplots(figsize=(10, 6.5))
            ax.imshow(representative_lk['flow_rgb'])
            ax.set_title(f"Lucas-Kanade Pyramidal Optical Flow Tracks (Frame #{representative_lk['frame_num']})", fontweight='bold')
            ax.axis('off')
            plt.tight_layout()
            fig.savefig(os.path.join(TASK4_DIR, "3_lucas_kanade_optical_flow.png"), bbox_inches='tight', dpi=150)
            plt.show()
    else:
        cap.release()
        print("[!] Unable to initialize optical flow on video stream.")
else:
    print("[!] Task 7.4 Pending: No video file found.")
    print("    Upload a video (.mp4, .avi) to 'data/videos/' and re-run this cell to generate Optical Flow outputs.")


In [ ]:
# Cell 7.5: Motion Estimation Method Comparison
if video_meta is not None:
    motion_comp_records = [
        {
            "Method": "Temporal Frame Differencing",
            "Core Mathematical Principle": "Consecutive temporal subtraction |I_t - I_{t-1}| > T",
            "Output Representation": "Binary motion change mask",
            "FPS Throughput": fd_fps if 'fd_fps' in locals() else "N/A",
            "Computational Complexity": "O(N) Pixel-wise (Fastest)",
            "Key Strengths": "Extremely fast; minimal memory; no initialization phase required.",
            "Key Limitations": "Generates double silhouettes (ghosting); blind to stationary objects; sensitive to camera shake."
        },
        {
            "Method": "GMM Background Subtraction (MOG2)",
            "Core Mathematical Principle": "Adaptive multimodal Gaussian Mixture per pixel with shadow modeling",
            "Output Representation": "Segmented foreground mask + bounding boxes",
            "FPS Throughput": mog_fps if 'mog_fps' in locals() else "N/A",
            "Computational Complexity": "O(N * K) Multi-Gaussian updating",
            "Key Strengths": "Segments complete foreground bodies; adapts to gradual light changes; suppresses shadows.",
            "Key Limitations": "Requires static camera; background learning lag; temporary ghosting when background objects move."
        },
        {
            "Method": "Lucas-Kanade Sparse Optical Flow",
            "Core Mathematical Principle": "Spatial-temporal gradient constraint: I_x*u + I_y*v + I_t = 0 over image pyramids",
            "Output Representation": "Motion displacement vectors (u, v) on corners",
            "FPS Throughput": lk_fps if 'lk_fps' in locals() else "N/A",
            "Computational Complexity": "O(P * W^2) per tracked feature point",
            "Key Strengths": "Quantifies precise velocity magnitude and direction; functions under moving camera platforms.",
            "Key Limitations": "Assumes brightness constancy and small displacements; fails in textureless regions (aperture problem)."
        }
    ]
    
    df_task4_comp = pd.DataFrame(motion_comp_records)
    csv_comp_task4 = os.path.join(METRICS_DIR, "task4_motion_metrics.csv")
    df_task4_comp.to_csv(csv_comp_task4, index=False)
    
    print("=" * 75)
    print("TASK 4 MOTION ESTIMATION COMPARATIVE BENCHMARK:")
    display(df_task4_comp)
    print("=" * 75)
    print(f"[✓] Motion estimation benchmark exported to: {csv_comp_task4}")
else:
    print("[Pending] Task 7.5 awaiting video input.")


---
## Section 8: Comprehensive Results and Metric Summary
Aggregates and verifies the metric tables exported across all four tasks.


In [ ]:
# Cell 8.1: Unified Results Dashboard and Metric File Verification
print("=" * 70)
print("COMPREHENSIVE EXPERIMENTAL RESULTS SUMMARY:")
print("=" * 70)

metric_csvs = [
    ("Task 1: Restoration Metrics", os.path.join(METRICS_DIR, "task1_restoration_metrics.csv")),
    ("Task 2: Feature Invariance Metrics", os.path.join(METRICS_DIR, "task2_feature_metrics.csv")),
    ("Task 3: Object Recognition Metrics", os.path.join(METRICS_DIR, "task3_detection_metrics.csv")),
    ("Task 4: Motion Estimation Metrics", os.path.join(METRICS_DIR, "task4_motion_metrics.csv")),
]

for title, path in metric_csvs:
    print(f"\n--- {title} ---")
    if os.path.exists(path):
        df_loaded = pd.read_csv(path)
        display(df_loaded)
    else:
        print(f"[Status: Pending Execution] '{os.path.basename(path)}' not yet generated.")
        print("  Reason: Awaiting user image/video data upload.")
print("=" * 70)


---
## Section 9: Generated Artifacts Inventory
Lists and verifies all visual plots and metrics exported to the outputs/ directory.


In [ ]:
# Cell 9.1: Inventory of Generated Output Files
print("=" * 70)
print("INVENTORY OF GENERATED OUTPUT ARTIFACTS:")
print("=" * 70)

all_outputs = []
for root, dirs, files in os.walk(OUTPUT_DIR):
    for f in sorted(files):
        if f == '.gitkeep':
            continue
        full_p = os.path.join(root, f)
        rel_p = os.path.relpath(full_p, BASE_DIR)
        size_kb = os.path.getsize(full_p) / 1024
        all_outputs.append((rel_p, size_kb))

if all_outputs:
    for rel_p, size_kb in all_outputs:
        print(f"  [Artifact] {rel_p:<55} ({size_kb:.1f} KB)")
    print(f"\nTotal Artifacts Generated: {len(all_outputs)}")
else:
    print("  [Notice] No generated outputs found yet.")
    print("  Run preceding task cells after providing input data in 'data/images/' and 'data/videos/'.")
print("=" * 70)
